In [ ]:
# V4 FINAL
import os
import shutil
import random
import datetime 
from datetime import timedelta 
import pandas as pd
from docx import Document
from docx2pdf import convert
from PyPDF2 import PdfReader, PdfWriter, PdfMerger
import fitz  # PyMuPDF

# ==============================================================================
# 0. FUNCIONES AUXILIARES
# ==============================================================================
MESES_INGLES = ["", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

def formatear_fecha_hora(dt):
    dia = dt.day
    mes = MESES_INGLES[dt.month]
    anio = dt.year
    hora = dt.strftime("%I:%M %p").lstrip("0")
    return f"{dia}-{mes}-{anio} {hora} EST"

def generar_tiempos(fecha_base_str):
    try:
        dt_base = datetime.datetime.strptime(fecha_base_str, "%m/%d/%Y")
    except ValueError:
        return None
    
    hora_inicio = random.randint(14, 17)
    minuto_inicio = random.randint(0, 59)
    dt1 = dt_base.replace(hour=hora_inicio, minute=minuto_inicio)
    dt2 = dt1 + timedelta(minutes=20)
    dt3 = dt2 + timedelta(minutes=3)
    return {
        "Process started": formatear_fecha_hora(dt1),
        "Document viewed": formatear_fecha_hora(dt2),
        "Document accepted & signed": formatear_fecha_hora(dt3),
        "Document has been completed": formatear_fecha_hora(dt3)
    }

def concatenar_fila(row):
    valores_no_nan = [str(valor) for valor in row[['address1', 'address2', 'others']] if pd.notna(valor)]
    return ', '.join(valores_no_nan)

# ==============================================================================
# 1. CONFIGURACIÓN INICIAL
# ==============================================================================
root_dir = "D:\\Downloads\\Medical_Requests"
os.chdir(root_dir)

list_direct = os.listdir()
nombreFA = ""
nombreWord = ""

for archivo in list_direct:
    if "pdf" in archivo.lower():
        nombreFA = archivo
    elif "docx" in archivo.lower():
        nombreWord = archivo

if not nombreFA:
    print("❌ No se encontró ningún archivo PDF en la carpeta raíz.")
    exit()

input_pdf_path = os.path.join(root_dir, nombreFA)

print("\n=== SISTEMA DE SOLICITUDES MÉDICAS AUTOMATIZADO ===")
print(f"📄 Archivo PDF detectado: {nombreFA}")

# Solicitud de Datos
try:
    page_number_1 = int(input("Ingrese el número de la PRIMERA página a extraer: "))
    page_number_2 = int(input("Ingrese el número de la SEGUNDA página a extraer: "))
except ValueError:
    print("❌ Error: Ingrese solo números enteros.")
    exit()

folder_name = input("Ingrese el nombre del cliente: ")
if not folder_name:
    print("❌ Error: Debe ingresar un nombre.")
    exit()

print("\n--- Datos para el llenado ---")
fecha_inicio = input("Fecha INICIO (ej. 10/27/2025): ")
fecha_solicitud = input("Fecha FINAL (ej. 01/16/2026): ")
fecha_firma = input("Fecha de FIRMA (ej. 01/16/2026): ") 
email_correcto = "Gabriel@krompecherlaw.com"

nuevas_fechas_audit = generar_tiempos(fecha_firma)
if not nuevas_fechas_audit:
    print("❌ Error en formato de fecha de firma.")
    exit()

# ==============================================================================
# 2. CREACIÓN DE CARPETA Y PREPARACIÓN DE ARCHIVOS
# ==============================================================================
folder_path = os.path.join(root_dir, folder_name)
try:
    os.mkdir(folder_path)
    print(f"\n✅ Carpeta creada: {folder_path}")
except OSError:
    print(f"\n⚠️ La carpeta '{folder_name}' ya existe, continuando...")

if nombreWord:
    destino_word = os.path.join(folder_path, nombreWord)
    if not os.path.exists(destino_word):
        shutil.move(os.path.join(root_dir, nombreWord), folder_path)
else:
    print("⚠️ Advertencia: No se encontró archivo Word (.docx)")

archivo_excel_origen = "Medical_request_facilities.xlsx"
if os.path.exists(archivo_excel_origen):
    shutil.copy(archivo_excel_origen, folder_path)
else:
    print("❌ ERROR: No se encuentra 'Medical_request_facilities.xlsx' en Descargas.")
    exit()

hipaa_pdf_path = os.path.join(folder_path, "HIPAA.pdf")

# ==============================================================================
# 3. GENERAR Y EDITAR HIPAA MAESTRO (Páginas 1 y 2)
# ==============================================================================
print(f">> Extrayendo páginas de: {input_pdf_path}")

try:
    with open(input_pdf_path, "rb") as file:
        pdf_reader = PdfReader(file)
        pdf_writer = PdfWriter()
        total_pages = len(pdf_reader.pages)
        if (0 < page_number_1 <= total_pages) and (0 < page_number_2 <= total_pages):
            pdf_writer.add_page(pdf_reader.pages[page_number_1 - 1])
            pdf_writer.add_page(pdf_reader.pages[page_number_2 - 1])
            with open(hipaa_pdf_path, 'wb') as output_file:
                pdf_writer.write(output_file)
        else:
            print("❌ Error: Números de página inválidos.")
            exit()
except FileNotFoundError:
    print(f"❌ Error Crítico: No se encuentra el archivo original '{nombreFA}'.")
    exit()

# Edición General (Fechas, Firma, Email, Audit Trail)
print(">> Editando HIPAA Maestro (Fechas, Firma, Audit Trail)...")
doc = fitz.open(hipaa_pdf_path)
page1 = doc[0]

# Through dates
text_instances = page1.search_for("through")
if text_instances:
    rect = text_instances[0]
    page1.draw_rect(fitz.Rect(rect.x0 - 60, rect.y0, rect.x0, rect.y1 - 2), color=(1,1,1), fill=(1,1,1))
    ancho_texto = fitz.get_text_length(fecha_inicio, fontsize=10)
    page1.insert_text((rect.x0 - ancho_texto - 5, rect.y1 - 2), fecha_inicio, fontsize=10, color=(0,0,0))
    page1.insert_text((rect.x1 + 5, rect.y1 - 2), fecha_solicitud, fontsize=10, color=(0,0,0))

# Email
email_label = page1.search_for("Email:")
if email_label:
    label_rect = email_label[0]
    page1.draw_rect(fitz.Rect(label_rect.x1 + 2, label_rect.y0, label_rect.x1 + 250, label_rect.y1 + 2), color=(1,1,1), fill=(1,1,1))
    page1.insert_text((label_rect.x1 + 5, label_rect.y1 - 2), email_correcto, fontsize=10, fontname="hebo", color=(0,0,0))

# Firma
date_labels = page1.search_for("Date:")
if len(date_labels) >= 2:
    target_rect = date_labels[-2]
    page1.draw_rect(fitz.Rect(target_rect.x1 + 2, target_rect.y0 - 10, target_rect.x1 + 200, target_rect.y1 + 10), color=(1,1,1), fill=(1,1,1))
    page1.insert_text((target_rect.x1 + 5, target_rect.y1 - 2), fecha_firma, fontsize=10, color=(0,0,0))

# Pagina 2 (Audit Trail)
page2 = doc[1]
labels_audit = ["Process started", "Document viewed", "Document accepted & signed", "Document has been completed"]
for label in labels_audit:
    text_instances = page2.search_for(label)
    if text_instances:
        rect_label = text_instances[0]
        zona_analisis = fitz.Rect(rect_label.x0, rect_label.y1, rect_label.x0 + 350, rect_label.y1 + 35)
        texto_zona = page2.get_text("text", clip=zona_analisis)
        reference_id_rescate = None
        for linea in texto_zona.split('\n'):
            if "Reference ID:" in linea:
                reference_id_rescate = linea.strip()
                break
        
        altura_borrado = 28 if reference_id_rescate else 14
        redact_rect = fitz.Rect(rect_label.x0, rect_label.y1, rect_label.x0 + 300, rect_label.y1 + altura_borrado)
        page2.add_redact_annot(redact_rect, fill=(1, 1, 1))
        page2.apply_redactions()
        y_fecha = rect_label.y1 + 10
        page2.insert_text((rect_label.x0, y_fecha), nuevas_fechas_audit[label], fontsize=9, fontname="helv", color=(0, 0, 0))
        if reference_id_rescate:
            page2.insert_text((rect_label.x0, y_fecha + 11), reference_id_rescate, fontsize=9, fontname="helv", color=(0, 0, 0))

temp_output = os.path.join(folder_path, "HIPAA_temp.pdf")
doc.save(temp_output)
doc.close()
os.remove(hipaa_pdf_path)
os.rename(temp_output, hipaa_pdf_path)

if os.path.exists(input_pdf_path):
    shutil.move(input_pdf_path, folder_path)

# ==============================================================================
# 4. PROCESO DE FACILITIES (EXCEL + WORD + MERGE + FACILITY NAME)
# ==============================================================================
print("\n=== INICIANDO PROCESO POR FACILITY ===")
os.chdir(folder_path)

try:
    df = pd.read_excel("Medical_request_facilities.xlsx")
    facilities = df["name"]
except Exception as e:
    print(f"❌ Error leyendo Excel: {e}")
    exit()

for facility in facilities:
    print(f"🔄 Procesando: {facility}")
    
    try:
        if not os.path.exists(str(facility)):
            os.mkdir(str(facility))
    except OSError:
        pass
        
    facility_path = os.path.join(os.getcwd(), str(facility))
    shutil.copy("HIPAA.pdf", facility_path)
    shutil.copy(nombreWord, facility_path)
    
    os.chdir(facility_path)
    
    # --------------------------------------------------------------------------
    # NUEVA SECCIÓN: INSERTAR "FACILITY NAME" EN EL PDF
    # --------------------------------------------------------------------------
    try:
        doc_hipaa_fac = fitz.open("HIPAA.pdf")
        page_fac = doc_hipaa_fac[0]
        
        # Buscar "Facility Name:"
        fac_label = page_fac.search_for("Facility Name:")
        
        if fac_label:
            rect_fac = fac_label[0]
            # Insertar el nombre del facility justo a la derecha
            # Sumamos un pequeño margen a x1
            x_pos = rect_fac.x1 + 5
            y_pos = rect_fac.y1 - 2
            
            # Limpieza opcional (dibujar cuadro blanco por si hubiera algo)
            # clean_rect = fitz.Rect(x_pos, rect_fac.y0, x_pos + 300, rect_fac.y1 + 2)
            # page_fac.draw_rect(clean_rect, color=(1,1,1), fill=(1,1,1))
            
            page_fac.insert_text(
                (x_pos, y_pos), 
                str(facility), 
                fontsize=10, 
                color=(0,0,0)
            )
            
            # Guardar cambios en el PDF de esta carpeta
            doc_hipaa_fac.save("HIPAA_temp_fac.pdf")
            doc_hipaa_fac.close()
            os.remove("HIPAA.pdf")
            os.rename("HIPAA_temp_fac.pdf", "HIPAA.pdf")
            # print("   -> Nombre de facility añadido al HIPAA.")
        else:
            print("   ⚠️ Advertencia: No se encontró 'Facility Name:' en el PDF.")
            doc_hipaa_fac.close()
            
    except Exception as e:
        print(f"   ❌ Error editando PDF facility: {e}")

    # --------------------------------------------------------------------------
    # FIN NUEVA SECCIÓN
    # --------------------------------------------------------------------------
    
    try:
        doc_word = Document(nombreWord)
        
        # Eliminar Fax
        word_to_find = "Fax"
        for paragraph in doc_word.paragraphs:
            if word_to_find in paragraph.text:
                paragraph.text = paragraph.text.replace(word_to_find, "")

        # Reemplazar through present
        text_to_find_present = "through present"
        new_text_present = f"through {fecha_solicitud}"
        for paragraph in doc_word.paragraphs:
            if text_to_find_present in paragraph.text:
                paragraph.text = paragraph.text.replace(text_to_find_present, new_text_present)

        # Insertar Via y Dirección
        word_to_find_via = "Via :"
        fila_especifica = df[df['name'] == facility]
        
        if not fila_especifica.empty:
            value_to_add = fila_especifica.apply(concatenar_fila, axis=1).iloc[0]
            for paragraph in doc_word.paragraphs:
                if word_to_find_via in paragraph.text:
                    index = paragraph.text.find(word_to_find_via)
                    paragraph.text = paragraph.text[:index + len(word_to_find_via)] + " " + value_to_add + " " + paragraph.text[index+len(word_to_find_via):]
            
            for paragraph in doc_word.paragraphs:
                if word_to_find_via in paragraph.text:
                    for run in paragraph.runs:
                        if word_to_find_via in run.text:
                            run.font.bold = True
        else:
            print(f"⚠️  No se encontraron datos en Excel para: {facility}")

        doc_word.save("Modified request.docx")
        
        # Convertir y Unir
        convert("Modified request.docx", "Modified request.pdf")
        
        nombre_archivo_final = f" MR-MB_Request_DONE {facility}.pdf"
        fecha_actual_str = datetime.datetime.now().strftime("%Y.%m.%d")
        output_pdf_final = fecha_actual_str + nombre_archivo_final
        
        merger = PdfMerger()
        if os.path.exists("Modified request.pdf"): merger.append("Modified request.pdf")
        if os.path.exists("HIPAA.pdf"): merger.append("HIPAA.pdf")
                
        merger.write(output_pdf_final)
        merger.close()
        print(f"✅ Generado: {output_pdf_final}")

    except Exception as e:
        print(f"❌ Error procesando WORD/MERGE {facility}: {e}")
    
    os.chdir(folder_path)

print("\n✨ ¡TODO EL PROCESO COMPLETADO EXITOSAMENTE!")


=== SISTEMA DE SOLICITUDES MÉDICAS AUTOMATIZADO ===
📄 Archivo PDF detectado: FA Signed Marcelino Garcia Hernandez - E.pdf

--- Datos para el llenado ---

✅ Carpeta creada: D:\Downloads\Medical_Requests\Marcelino Garcia Hernandez
>> Extrayendo páginas de: D:\Downloads\Medical_Requests\FA Signed Marcelino Garcia Hernandez - E.pdf
>> Editando HIPAA Maestro (Fechas, Firma, Audit Trail)...

=== INICIANDO PROCESO POR FACILITY ===
🔄 Procesando: Change HealthCare TES


  0%|          | 0/1 [00:00<?, ?it/s]

✅ Generado: 2026.02.10 MR-MB_Request_DONE Change HealthCare TES.pdf
🔄 Procesando: Wake Emergency Physicians


  0%|          | 0/1 [00:00<?, ?it/s]

✅ Generado: 2026.02.10 MR-MB_Request_DONE Wake Emergency Physicians.pdf
🔄 Procesando: Wake Radiology


...

✅ Generado: 2026.02.10 MR-MB_Request_DONE Wake Radiology.pdf
🔄 Procesando: WakeMed Health and Hospitals


  0%|          | 0/1 [00:00<?, ?it/s]

✅ Generado: 2026.02.10 MR-MB_Request_DONE WakeMed Health and Hospitals.pdf
🔄 Procesando: WakeOrthopaedics


  0%|          | 0/1 [00:00<?, ?it/s]

✅ Generado: 2026.02.10 MR-MB_Request_DONE WakeOrthopaedics.pdf

✨ ¡TODO EL PROCESO COMPLETADO EXITOSAMENTE!
